# Project 1 â€” Supply Chain Demand Experimentation Notebook

This notebook is a proper experimentation notebook for the supply-chain project. It starts with data cleaning, repairs missing values where the measurement is recoverable, and then compares several forecasters on the same holdout split before selecting one candidate for the production pipeline.

## Data cleaning and NaN policy

The main reason to preserve a `NaN` is that it can mean the true business fact is not observable, not that the value is zero. In this data, `units_sold` and `closing_stock` gaps can be repaired by interpolation or by the inventory balance identity:

`closing_stock[t] = closing_stock[t-1] + units_received[t] - units_sold[t]`

That is safer than forcing zero because zero changes the business meaning of the event. If a row truly cannot be reconstructed because the calendar or SKU history is missing, we keep the missing value flagged rather than inventing a value.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RAW_PATH = Path("project1_supply_chain_demand.csv")
NEW_SKUS = {"SKU-2000", "SKU-2001", "SKU-2002"}
HOLDOUT_DAYS = 21
print("Setup complete.")


Setup complete.


In [13]:
df = pd.read_csv(RAW_PATH, parse_dates=["date"])
print(f"Raw rows: {len(df)}")

# Normalize category casing
old_n = len(df)
df = df.drop_duplicates(subset=["date", "sku_id"], keep="first")
print(f"Duplicate rows dropped: {old_n - len(df)}")
df["category"] = df["category"].str.strip().str.title()

# Lead time per SKU use the mode value to keep a trusted, consistent lead-time signal.
lead_time_map = df.groupby("sku_id")["lead_time_days"].agg(lambda s: s.mode().iloc[0]).to_dict()

cleaned_rows = []
for sku, g in df.groupby("sku_id"):
    g = g.set_index("date").sort_index()
    full_idx = pd.date_range(g.index.min(), g.index.max(), freq="D")
    g = g.reindex(full_idx)
    g["sku_id"] = sku
    g["category"] = g["category"].ffill().bfill()
    g["lead_time_days"] = lead_time_map[sku]
    g["units_received"] = g["units_received"].fillna(0)
    g["units_sold"] = g["units_sold"].interpolate(method="linear").bfill().ffill()
    stock = g["closing_stock"].copy()
    for i in range(1, len(stock)):
        if pd.isna(stock.iloc[i]) and not pd.isna(stock.iloc[i - 1]):
            implied = stock.iloc[i - 1] + g["units_received"].iloc[i] - g["units_sold"].iloc[i]
            stock.iloc[i] = max(implied, 0)
    stock = stock.interpolate(method="linear").bfill().ffill()
    g["closing_stock"] = stock
    cleaned_rows.append(g)

clean = pd.concat(cleaned_rows).rename_axis("date").reset_index()
clean = clean.sort_values(["sku_id", "date"]).reset_index(drop=True)
clean["dow"] = clean["date"].dt.dayofweek
print(clean.head())


Raw rows: 4551
Duplicate rows dropped: 15
        date    sku_id category  units_sold  units_received  closing_stock  \
0 2026-01-01  SKU-1000   Snacks       110.0               0          403.0   
1 2026-01-02  SKU-1000   Snacks       278.0               0          125.0   
2 2026-01-03  SKU-1000   Snacks       121.0             903          907.0   
3 2026-01-04  SKU-1000   Snacks        75.0               0          832.0   
4 2026-01-05  SKU-1000   Snacks        71.0               0          761.0   

   lead_time_days  dow  
0               7    3  
1               7    4  
2               7    5  
3               7    6  
4               7    0  


In [14]:
for w in (7, 14, 28):
    clean[f"roll_mean_{w}"] = clean.groupby("sku_id")["units_sold"].transform(
        lambda s: s.shift(1).rolling(w, min_periods=1).mean()
    )
    clean[f"roll_std_{w}"] = clean.groupby("sku_id")["units_sold"].transform(
        lambda s: s.shift(1).rolling(w, min_periods=2).std()
    )

clean["lag_1"] = clean.groupby("sku_id")["units_sold"].shift(1)
clean["lag_7"] = clean.groupby("sku_id")["units_sold"].shift(7)

cat_daily = clean.groupby(["category", "date"])["units_sold"].mean().rename("cat_avg_demand")
clean = clean.merge(cat_daily, on=["category", "date"], how="left")
clean["cat_roll_mean_14"] = clean.groupby("category")["cat_avg_demand"].transform(
    lambda s: s.shift(1).rolling(14, min_periods=1).mean()
)

FEATURES = [
    "dow", "roll_mean_7", "roll_mean_14", "roll_mean_28",
    "roll_std_7", "roll_std_14", "roll_std_28",
    "lag_1", "lag_7", "cat_roll_mean_14", "lead_time_days",
]
CAT_FEATURES = ["category"]

model_df = clean.dropna(subset=["roll_mean_7", "lag_1"]).copy()
for c in CAT_FEATURES:
    model_df[c] = model_df[c].astype("category")

established = model_df[~model_df.sku_id.isin(NEW_SKUS)]
cutoff = established["date"].max() - pd.Timedelta(days=HOLDOUT_DAYS)
train = established[established["date"] <= cutoff]
test = established[established["date"] > cutoff]

X_train = pd.get_dummies(train[FEATURES + CAT_FEATURES], columns=CAT_FEATURES, dtype=float)
X_test = pd.get_dummies(test[FEATURES + CAT_FEATURES], columns=CAT_FEATURES, dtype=float)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print(X_train.shape, X_test.shape)


(3950, 16) (525, 16)


## Model comparison experiment

Compare several learners on the same time-based holdout and WAPE/MAE metric. The candidate that performs best becomes the model exported to the production pipeline.


In [ ]:
def wape(y_true, y_pred):
    return np.abs(y_true - y_pred).sum() / np.abs(y_true).sum()

candidates = {
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=140, max_depth=8, min_samples_leaf=2,
        random_state=42,
    ),
    "ExtraTreesRegressor": ExtraTreesRegressor(
        n_estimators=140, max_depth=8, min_samples_leaf=2,
        random_state=42,
    ),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
        max_iter=80, learning_rate=0.05, max_depth=4,
        random_state=42,
    ),
}

results = []
for name, estimator in candidates.items():
    estimator.fit(X_train, train['units_sold'])
    pred = np.clip(estimator.predict(X_test), 0, None)
    w = wape(test['units_sold'], pred)
    ma = mean_absolute_error(test['units_sold'], pred)
    results.append({"model": name, "wape": float(w), "mae": float(ma)})
    print(f"{name}: WAPE={w:.4f} | MAE={ma:.2f}")

experiment_df = pd.DataFrame(results).sort_values("wape", ascending=True)
print("Notebook experiment ranking:")
print(experiment_df.to_string(index=False))

best_model_name = experiment_df.iloc[0]['model']
print(f"Selected model for pipeline: {best_model_name}")


RandomForestRegressor: WAPE=0.1907 | MAE=57.23
ExtraTreesRegressor: WAPE=0.1797 | MAE=53.94
HistGradientBoostingRegressor: WAPE=0.1806 | MAE=54.20
Notebook experiment ranking:
                        model     wape       mae
          ExtraTreesRegressor 0.179702 53.939271
HistGradientBoostingRegressor 0.180573 54.200712
        RandomForestRegressor 0.190682 57.234858
Selected model for pipeline: ExtraTreesRegressor


In [16]:
experiment_df.to_json("./output/p1_experiment_ranking.json", orient="records", indent=2)
print("Wrote p1_experiment_ranking.json")


Wrote p1_experiment_ranking.json
